## 构建用于网页浏览和摘要的自带浏览器（BYOB）工具

**免责声明：本 cookbook 仅用于教学目的。在使用网页搜索和抓取技术时，请确保遵守所有适用的法律和服务条款。本 cookbook 会将搜索范围限制在 openai.com 域名内，检索公开信息以说明相关概念。**

GPT-4o 等大语言模型（LLM）存在知识截止日期，这意味着它们缺少该时间点之后所发生事件的信息。在最新数据至关重要的场景中，有必要让 LLM 能够访问当前的网络信息，以确保回答准确且相关。

在本指南中，我们将使用 Python 构建一个自带浏览器（Bring Your Own Browser，BYOB）工具来克服这一限制。我们的目标是创建一个能在你的应用中提供最新答案的系统，包括最新进展，例如 OpenAI 最近发布的产品。通过将网页搜索能力与 LLM 集成，我们将使模型能够基于网上可获得的最新信息生成回答。

虽然你可以使用任何公开可用的搜索 API，但我们将使用 Google 的 Custom Search API 来执行网页搜索。从搜索结果中检索到的信息将被处理后传给 LLM，通过检索增强生成（Retrieval-Augmented Generation，RAG）生成最终回答。

**自带浏览器（BYOB）** 工具允许用户以编程方式执行网页浏览任务。在本 notebook 中，我们将创建一个 BYOB 工具，它会：

**#1. 设置搜索引擎：** 使用公共搜索 API（例如 Google 的 Custom Search API）执行网页搜索，并获取相关搜索结果列表。  

**#2. 构建搜索字典：** 从搜索结果中收集每个网页的标题、URL 和摘要，创建结构化的信息字典。  

**#3. 生成 RAG 回答：** 实现检索增强生成（RAG），将收集到的信息传给 LLM，由它针对用户的查询生成最终回答。



### 用例 
在本 cookbook 中，我们以一个想按时间顺序列出 OpenAI 近期产品发布的用户为例。由于当前的 GPT-4o 模型存在知识截止日期，预计模型不会知道近期的产品发布，例如 2024 年 9 月发布的 o1-preview 模型。 


In [1]:
from openai import OpenAI

client = OpenAI()

search_query = "List the latest OpenAI product launches in chronological order from latest to oldest in the past 2 years"


response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful agent."},
        {"role": "user", "content": search_query}]
).choices[0].message.content

print(response)

OpenAI has had several notable product launches and updates in the past couple of years. Here’s a chronological list of some significant ones:

1. **ChatGPT (November 2022)**: OpenAI launched an AI chatbot called ChatGPT, which is based on GPT-3.5. This chatbot became widely popular due to its impressive capabilities in understanding and generating human-like text.

2. **GPT-4 (March 2023)**: OpenAI released GPT-4, the latest version of their Generative Pre-trained Transformer model. It brought improvements in both performance and accuracy over its predecessors.

3. **DALL-E 2 (April 2022)**: The second version of DALL-E, an AI model that can generate images from textual descriptions, was launched with enhanced image resolution and more robust capabilities.

4. **Whisper (September 2022)**: Whisper, an automatic speech recognition (ASR) system, was introduced. This model can handle multiple languages and is useful for transcribing and understanding spoken language.

These are some of t

由于知识截止日期的限制，正如预期的那样，模型并不知道 OpenAI 近期发布的产品。

### 设置 BYOB 工具
为了向模型提供近期事件的信息，我们将按以下步骤进行：

##### 第 1 步：设置搜索引擎以提供网页搜索结果
##### 第 2 步：构建包含网页标题、URL 和摘要的搜索字典
##### 第 3 步：将信息传给模型，针对用户查询生成 RAG 回答  


开始之前，请确保满足以下条件：机器上已安装 **Python 3.12 或更高版本**。你还需要一个 Google Custom Search API 密钥和自定义搜索引擎 ID（CSE ID）。已安装必要的 Python 包：`requests`、`beautifulsoup4`、`openai`。并确保已将 OPENAI_API_KEY 设置为环境变量。

#### 第 1 步：设置搜索引擎以提供网页搜索结果
你可以使用任何公开可用的网页搜索 API 来完成这项任务。我们将使用 Google 的 Custom Search API 配置一个自定义搜索引擎。该引擎会根据用户的查询获取相关网页列表，重点获取最新且最相关的结果。  

**a. 配置搜索 API 密钥和函数：** 从 Google Developers Console 获取 Google API 密钥和自定义搜索引擎 ID（CSE ID）。你可以前往这个 [Programmable Search Engine 链接](https://developers.google.com/custom-search/v1/overview) 设置 API 密钥以及自定义搜索引擎 ID（CSE ID）。 

下面的 `search` 函数根据搜索词、API 密钥和 CSE ID 以及要返回的搜索结果数量来设置搜索。我们将引入一个参数 `site_filter`，将输出限制为仅来自 `openai.com`
  

In [2]:
import requests  # For making HTTP requests to APIs and websites

def search(search_item, api_key, cse_id, search_depth=10, site_filter=None):
    service_url = 'https://www.googleapis.com/customsearch/v1'

    params = {
        'q': search_item,
        'key': api_key,
        'cx': cse_id,
        'num': search_depth
    }

    try:
        response = requests.get(service_url, params=params)
        response.raise_for_status()
        results = response.json()

        # Check if 'items' exists in the results
        if 'items' in results:
            if site_filter is not None:
                
                # Filter results to include only those with site_filter in the link
                filtered_results = [result for result in results['items'] if site_filter in result['link']]

                if filtered_results:
                    return filtered_results
                else:
                    print(f"No results with {site_filter} found.")
                    return []
            else:
                if 'items' in results:
                    return results['items']
                else:
                    print("No search results found.")
                    return []

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the search: {e}")
        return []


**b. 确定搜索引擎的搜索词：** 在从第三方 API 检索特定结果之前，我们可能需要使用查询扩展来确定浏览器搜索 API 应检索的具体词语。**查询扩展**（Query expansion）是指通过添加相关词、同义词或变体来扩展原始用户查询的过程。这项技术非常重要，因为像 Google Custom Search API 这样的搜索引擎通常更擅长匹配一系列相关词语，而不仅仅是用户所使用的自然语言提示。 

例如，仅使用原始查询 `"List the latest OpenAI product launches in chronological order from latest to oldest in the past 2 years"` 进行搜索，返回的结果可能比使用 `"Latest OpenAI product launches"` 这样更具体、更直接的简洁短语进行搜索更少、相关性更低。在下面的代码中，我们将使用用户的原始 `search_query` 生成一个更具体的搜索词，用于通过 Google API 检索结果。 

In [3]:
search_term = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Provide a google search term based on search query provided below in 3-4 words"},
        {"role": "user", "content": search_query}]
).choices[0].message.content

print(search_term)

Latest OpenAI product launches


**c. 调用搜索函数：** 有了搜索词之后，我们将调用搜索函数从 Google 搜索 API 检索结果。此时结果只包含网页链接和一段摘录。在下一步中，我们将从网页中检索更多信息，并将其汇总到一个字典中传给模型。

In [4]:
from dotenv import load_dotenv
import os

load_dotenv('.env')

api_key = os.getenv('API_KEY')
cse_id = os.getenv('CSE_ID')

search_items = search(search_item=search_term, api_key=api_key, cse_id=cse_id, search_depth=10, site_filter="https://openai.com")


In [5]:
for item in search_items:
    print(f"Link: {item['link']}")
    print(f"Snippet: {item['snippet']}\n")

Link: https://openai.com/news/
Snippet: Overview ; Product. Sep 12, 2024. Introducing OpenAI o1 ; Product. Jul 25, 2024. SearchGPT is a prototype of new AI search features ; Research. Jul 18, 2024. GPT- ...

Link: https://openai.com/index/new-models-and-developer-products-announced-at-devday/
Snippet: Nov 6, 2023 ... GPT-4 Turbo with 128K context · We released the first version of GPT-4 in March and made GPT-4 generally available to all developers in July.

Link: https://openai.com/news/product/
Snippet: Discover the latest product advancements from OpenAI and the ways they're being used by individuals and businesses.

Link: https://openai.com/
Snippet: A new series of AI models designed to spend more time thinking before they respond. Learn more · (opens in a new window) ...

Link: https://openai.com/index/sora/
Snippet: Feb 15, 2024 ... We plan to include C2PA metadata(opens in a new window) in the future if we deploy the model in an OpenAI product. In addition to us developing ...



#### 第 2 步：构建包含网页标题、URL 和摘要的搜索字典
获得搜索结果后，我们将提取并整理相关信息，以便传给 LLM 生成最终输出。 

**a. 抓取网页内容：** 对于搜索结果中的每个 URL，获取网页并提取文本内容，同时过滤掉脚本和广告等无关数据，如函数 `retrieve_content` 所示。 

**b. 总结内容：** 使用 LLM 为抓取到的内容生成简洁的摘要，重点关注与用户查询相关的信息。可以向模型提供原始搜索文本，使其能够围绕搜索意图来总结内容，如函数 `summarize_content` 所示。 
  
**c. 创建结构化字典：** 将数据整理成一个字典或 DataFrame，其中包含每个网页的标题、链接和摘要。这一结构可以传给 LLM，用于生成带有适当引用的摘要。    


In [6]:
import requests
from bs4 import BeautifulSoup

TRUNCATE_SCRAPED_TEXT = 50000  # Adjust based on your model's context window
SEARCH_DEPTH = 5

def retrieve_content(url, max_tokens=TRUNCATE_SCRAPED_TEXT):
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')
            for script_or_style in soup(['script', 'style']):
                script_or_style.decompose()

            text = soup.get_text(separator=' ', strip=True)
            characters = max_tokens * 4  # Approximate conversion
            text = text[:characters]
            return text
        except requests.exceptions.RequestException as e:
            print(f"Failed to retrieve {url}: {e}")
            return None
        
def summarize_content(content, search_term, character_limit=500):
        prompt = (
            f"You are an AI assistant tasked with summarizing content relevant to '{search_term}'. "
            f"Please provide a concise summary in {character_limit} characters or less."
        )
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": content}]
            )
            summary = response.choices[0].message.content
            return summary
        except Exception as e:
            print(f"An error occurred during summarization: {e}")
            return None

def get_search_results(search_items, character_limit=500):
    # Generate a summary of search results for the given search term
    results_list = []
    for idx, item in enumerate(search_items, start=1):
        url = item.get('link')
        
        snippet = item.get('snippet', '')
        web_content = retrieve_content(url, TRUNCATE_SCRAPED_TEXT)
        
        if web_content is None:
            print(f"Error: skipped URL: {url}")
        else:
            summary = summarize_content(web_content, search_term, character_limit)
            result_dict = {
                'order': idx,
                'link': url,
                'title': snippet,
                'Summary': summary
            }
            results_list.append(result_dict)
    return results_list

In [7]:
results = get_search_results(search_items)

for result in results:
    print(f"Search order: {result['order']}")
    print(f"Link: {result['link']}")
    print(f"Snippet: {result['title']}")
    print(f"Summary: {result['Summary']}")
    print('-' * 80)

Search order: 1
Link: https://openai.com/news/
Snippet: Overview ; Product. Sep 12, 2024. Introducing OpenAI o1 ; Product. Jul 25, 2024. SearchGPT is a prototype of new AI search features ; Research. Jul 18, 2024. GPT- ...
Summary: OpenAI recently launched several notable products in 2024, including OpenAI o1 and SearchGPT, a prototype for enhanced AI search capabilities. Additionally, GPT-4o mini was introduced, enhancing cost-efficient intelligence. The organization also rolled out OpenAI for Nonprofits and ChatGPT Edu to support various sectors. Improvements in data analysis within ChatGPT and enhancements to the fine-tuning API were also announced. These updates reflect OpenAI's ongoing commitment to advancing AI technologies across different fields.
--------------------------------------------------------------------------------
Search order: 2
Link: https://openai.com/index/new-models-and-developer-products-announced-at-devday/
Snippet: Nov 6, 2023 ... GPT-4 Turbo with 128K conte

我们检索到了最新的结果。（注意，这些结果会因你执行此脚本的时间不同而有所变化。） 

#### 第 3 步：将信息传给模型，针对用户查询生成 RAG 回答
搜索数据已整理成 JSON 数据结构，我们将把这些信息连同原始用户查询一起传给 LLM，以生成最终回答。现在，LLM 的回答包含了其原始知识截止日期之后的信息，能够提供最新的见解。

In [8]:
import json 

final_prompt = (
    f"The user will provide a dictionary of search results in JSON format for search query {search_term} Based on on the search results provided by the user, provide a detailed response to this query: **'{search_query}'**. Make sure to cite all the sources at the end of your answer."
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": final_prompt},
        {"role": "user", "content": json.dumps(results)}],
    temperature=0

)
summary = response.choices[0].message.content

print(summary)

Based on the search results provided, here is a chronological list of the latest OpenAI product launches from the past two years, ordered from the most recent to the oldest:

1. **September 12, 2024**: **OpenAI o1**
   - A versatile AI tool designed to enhance reasoning capabilities.
   - Source: [OpenAI News](https://openai.com/news/)

2. **July 25, 2024**: **SearchGPT**
   - A prototype aimed at enhancing AI-driven search capabilities.
   - Source: [OpenAI News](https://openai.com/news/)

3. **July 18, 2024**: **GPT-4o mini**
   - A cost-efficient intelligence model.
   - Source: [OpenAI News](https://openai.com/news/)

4. **May 2024**: **OpenAI for Education**
   - Focuses on integrating AI into educational settings.
   - Source: [OpenAI News](https://openai.com/news/product/)

5. **February 15, 2024**: **Sora**
   - An AI model capable of generating high-quality text-to-video content.
   - Source: [OpenAI Sora](https://openai.com/index/sora/)

6. **November 6, 2023**: **GPT-4 Turbo

### 总结
 
大语言模型（LLM）存在知识截止日期，可能不了解近期发生的事件。为了向它们提供最新信息，你可以使用 Python 构建一个自带浏览器（BYOB）工具。该工具会检索当前的网络数据并提供给 LLM，从而实现最新的回答。

这个过程包括三个主要步骤：

**#1 设置搜索引擎：** 使用公共搜索 API（例如 Google 的 Custom Search API）执行网页搜索，并获取相关搜索结果列表。  

**#2 构建搜索字典：** 从搜索结果中收集每个网页的标题、URL 和摘要，创建结构化的信息字典。  

**#3. 生成 RAG 回答：** 实现检索增强生成（RAG），将收集到的信息传给 LLM，由它针对用户的查询生成最终回答。

按照这些步骤，你可以增强 LLM 在应用中提供最新答案的能力，包括最新进展，例如 OpenAI 最近发布的产品。